In [43]:
#!/usr/bin/env python3
import os, feedparser, textwrap
from dateutil import parser as dtparse
from pytz import timezone as pytz_timezone
from openai import OpenAI

# --- Config ---
RSS_FEEDS = ["https://www.smh.com.au/rss/business.xml"]
OUTPUT_CSV = "smh_business.csv"
MODEL = "gpt-4o"        # use gpt-4o-mini if you want cheaper dev
TEMPERATURE = 0.0
TZ = pytz_timezone("Australia/Sydney")
MAX_ITEMS = 35          # cap to avoid token overflow; bump if needed
PRINT_MODEL_OUTPUT = True

CSV_HEADER = "time,headline,sector,stock,article link"

SYSTEM_PROMPT = (
    "You are a CSV printer. Respond with CSV text only.\n"
    "Output MUST begin with EXACTLY this header on line 1:\n"
    "time,headline,sector,stock,article link\n"
    "Rules:\n"
    "1) Use provided time and headline as-is when available.\n"
    "2) Sector must be one of: Materials, Financials, Energy, Industrials, Health Care, "
    "Consumer Staples, Consumer Discretionary, Communication Services, Real Estate, Utilities, "
    "Information Technology, or Unknown.\n"
    "3) If a single ASX-listed company is clearly implied by headline/summary, set stock to its ASX ticker; "
    "else stock=UNMAPPED.\n"
    "4) Use only input fields; do not browse or invent.\n"
    "5) No commentary, no Markdown, no code fences. CSV only.\n"
    "6) If a field contains a comma or quote, wrap it in double quotes and escape inner quotes by doubling them."
)

def parse_time(entry):
    for attr in ("published", "updated"):
        val = getattr(entry, attr, None)
        if val:
            try:
                return dtparse.parse(val).astimezone(TZ).isoformat()
            except Exception:
                pass
    return ""

def collect_feed_items():
    items = []
    for url in RSS_FEEDS:
        feed = feedparser.parse(url)
        for e in feed.entries:
            items.append({
                "time": parse_time(e),
                "headline": getattr(e, "title", "").strip(),
                "summary": (getattr(e, "summary", "") or getattr(e, "description", "") or "").strip(),
                "link": getattr(e, "link", "").strip(),
            })
    # dedupe by link and clip to MAX_ITEMS
    seen, out = set(), []
    for it in items:
        if not it["link"] or it["link"] in seen:
            continue
        seen.add(it["link"])
        out.append(it)
        if len(out) >= MAX_ITEMS:
            break
    return out

def build_user_prompt(items):
    # Pass RSS fields verbatim; ask for one CSV row per article in same order
    parts = [
        "Produce CSV for these articles in the SAME order.",
        "Available fields per article: time, headline, summary, article link.",
        "Return CSV ONLY starting with the EXACT header:",
        CSV_HEADER,
        ""
    ]
    for i, it in enumerate(items, 1):
        parts.append(f"## ARTICLE {i}")
        parts.append(f"time: {it['time']}")
        parts.append(f"headline: {it['headline']}")
        parts.append(f"summary: {it['summary']}")
        parts.append(f"article link: {it['link']}")
        parts.append("")
    # Add one-shot mini example to force formatting
    example = textwrap.dedent(f"""\
        Example of exactly one row (illustrative only):
        {CSV_HEADER}
        2025-10-21T09:00:00+11:00,"Sample headline","Unknown","UNMAPPED","https://example.com/article"
    """)
    parts.append(example)
    parts.append("Now produce the CSV for the provided articles only.")
    return "\n".join(parts)

def call_model(prompt_text):
    client = OpenAI()
    resp = client.responses.create(
        model=MODEL,
        temperature=TEMPERATURE,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_text},
        ],
    )
    txt = resp.output_text
    if PRINT_MODEL_OUTPUT:
        print("\n--- RAW MODEL OUTPUT (first 800 chars) ---")
        print(txt[:800])
        print("\n--- END RAW ---\n")
    return txt

def ensure_csv_header(csv_text):
    # If model includes extra lines before the header, strip everything before the header
    lines = [ln for ln in csv_text.splitlines() if ln.strip()]
    if not lines:
        return CSV_HEADER + "\n"
    # find the header line
    try:
        idx = next(i for i, ln in enumerate(lines) if ln.strip() == CSV_HEADER)
        trimmed = lines[idx:]
        # Remove any duplicate header lines after the first
        out = [trimmed[0]] + [ln for ln in trimmed[1:] if ln.strip() != CSV_HEADER]
        return "\n".join(out).strip() + "\n"
    except StopIteration:
        # No header found: return only header
        return CSV_HEADER + "\n"

def save_csv(csv_text, path):
    cleaned = ensure_csv_header(csv_text)
    with open(path, "w", encoding="utf-8", newline="") as f:
        f.write(cleaned)
    rows = max(0, len([ln for ln in cleaned.splitlines()[1:] if ln.strip()]))
    print(f"Wrote {rows} rows to {path}")

def main():
    items = collect_feed_items()
    if not items:
        save_csv(CSV_HEADER + "\n", OUTPUT_CSV)
        return
    prompt = build_user_prompt(items)
    csv_text = call_model(prompt)

    # Retry once with a harsher instruction if header not present at all
    if CSV_HEADER not in csv_text:
        retry_prompt = prompt + "\n\nCRITICAL: Respond with CSV ONLY. First line must be exactly:\n" + CSV_HEADER
        csv_text = call_model(retry_prompt)

    save_csv(csv_text, OUTPUT_CSV)

if __name__ == "__main__":
    main()


2025-10-21 22:34:02 INFO HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"



--- RAW MODEL OUTPUT (first 800 chars) ---
time,headline,sector,stock,article link
2025-10-21T18:38:09+11:00,Rex Airlines rescued by US outfit Air T,Industrials,REX,https://www.smh.com.au/business/companies/rex-airlines-rescued-by-us-outfit-air-t-20251021-p5n41q.html?ref=rss&utm_medium=rss&utm_source=rss_business
2025-10-21T15:45:07+11:00,Trump’s critical minerals deal unearths a bonanza for Aussie miners,Materials,UNMAPPED,https://www.smh.com.au/business/companies/trump-s-critical-minerals-deal-unearths-a-bonanza-for-aussie-miners-20251021-p5n41a.html?ref=rss&utm_medium=rss&utm_source=rss_business
2025-10-21T15:35:10+11:00,Larvotto opens up new front with rich gold-antimony hits in NSW,Materials,LRV,https://www.smh.com.au/business/companies/larvotto-opens-up-new-front-with-rich-gold-antimony-hits-in-nsw-20251021-p5n47m.html?ref=rs

--- END RAW ---

Wrote 20 rows to smh_business.csv
